# 리포트 11 — 처리 사슬 — 직접파를 죽이고 표적을 세운다

> ECA 로 직접파를 지우고 CFAR 로 문턱을 세운다. **문턱을 어디에 두느냐가 결과를 정하므로** 그 교정을 먼저 적는다.

이 권은 아래 절로 이루어진다. 각 절은 **한 일 · 결과 · 방법 · 재현**을 자기 앞에 달고 있어, 필요한 절만 따로 읽어도 된다.

| 절 | 무엇을 말하나 |
|---|---|
| 1 | 수신 → ECA → 거리도플러 → CFAR, 사슬의 형상은 파형이 정한다 |
| 2 | 탭을 늘리면 환경이 정한 바닥에서 멈추고, 그 대가가 0-도플러 노치다 |
| 3 | 운용 형상에서 경험 Pfa 를 재니 명목값의 1.52~2.66 배였다 |
| 4 | 그 배율의 원인은 셀 상관이고, 교정표는 형상마다 다시 재야 한다 |

⭐ 숫자는 전부 계산 결과 JSON(원장)에서 주입된다 — 각주가 그 출처다. 원장이 다시 계산되면 빌더를 돌리는 것만으로 본문 숫자가 따라 바뀐다.


---

## 절 1. 수신 → ECA → 거리도플러 → CFAR, 사슬의 형상은 파형이 정한다



> ### 한 일
> **세 조명원을 하나의 동일한 검출 사슬에 물리고, 사슬의 각 단계가 파형마다 어떤 형상(거리 빈 수 · ECA 탭 수 · 도플러 빈)을 갖는지를 표로 고정했다.**

### 결과
1. 사슬은 네 단계다 — 2채널 수신 → ECA 로 직접파 제거 → 거리-도플러 상관 → CA-CFAR 판정. 각 단계는 앞 단계의 잔류물을 물려받는다.
2. 직접파 세기 DNR 은 WiFi 43.0 dB [^1] · LTE 60.0 dB [^2] · 5G 48.9 dB [^3] 다 — 수신단에서 가장 큰 신호이고 2단계가 지울 대상이다.
3. ECA 탭 수는 파형이 정한다 — WiFi 24 [^4] · LTE 14 [^5] · 5G 32 [^6].
4. 도플러 빈은 세 파형 모두 48 [^7]개다(CPI 당 프레임 수). 거리 빈 수와 PRF 는 파형마다 다르고, 아래 표가 그 형상을 한 자리에 모은다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 사슬 구현 | 네 단계가 한 파일에 있다 — `src/passive_process.py:42`(수신) · `:93,124`(ECA) · `:133`(거리-도플러) · `:153`(CA-CFAR) |
| 형상 표 | 선언값을 옮겨 적지 않고 검증 산출물 `outputs/verify_eca.json:meta.setups` 에서 직접 뽑는다 |
| 세 파형 동일 사슬 | 코드 경로가 하나다 — 파형이 바꾸는 것은 형상 파라미터뿐이다 |

### 재현

```bash
cd /home/yunjung/workspace/sionna2
PYTHONPATH=src ~/.venvs/py312/bin/python benchmark/verify_eca.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/viz_report04_detector.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/build_part09_detector.py
```

| | |
|---|---|
| 출력 | `outputs/verify_eca.json` |
| 소요 | ECA 검증 · 그림은 각각 수 분 (GPU 1장) |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| [편 44 «상시이면서 내용을 미리 아는 신호는 표준마다…»](44_illuminators.ipynb) | 세 표준의 상시 기준신호와 대역 |
| [편 47 «바이스태틱 거리 분해능은 c/B»](47_range-convention.ipynb) | 거리 규약 $\Delta R_b = c/B_{ref}$ |

---


## 수신 신호가 판정이 되기까지

패시브 검출은 네 단계다. 각 단계는 앞 단계의 잔류물을 물려받는다.

| 단계 | 하는 일 | 코드 |
|---|---|---|
| 1. 수신 | 서베일런스(표적 쪽을 보는 채널) + 레퍼런스(조명원을 직접 받는 채널) 2채널 | `src/passive_process.py:42` |
| 2. ECA | 직접파를 서베일런스에서 투영 제거 | `src/passive_process.py:93,124` |
| 3. 거리-도플러(CAF) | 레퍼런스와 지연 · 도플러 상관 | `src/passive_process.py:133` |
| 4. CA-CFAR | 이웃 셀로 문턱을 세우고 판정 | `src/passive_process.py:153` |


![f1_chain](../outputs/figures/report04_f1_chain.png)

**그림 1.** 수신 신호는 어떤 단계를 거쳐 검출 판정이 되는가?


## 사슬의 형상 — 파형이 정하는 것

직접파는 수신단에서 가장 큰 신호다. 그 크기가 DNR 이고, 2단계가 지울 대상이다. 거리 빈 수와 ECA 탭 수는 파형이 정하고, 도플러 빈은 세 파형 모두 48 [^7]개다.

| 파형 | DNR | ECA 탭 | 거리 빈 | PRF | Δf_d |
|---|---|---|---|---|---|
| WiFi 80MHz | 43.0 dB | 24 | 16 | 1000 Hz | 20.83 Hz |
| LTE 20MHz | 60.0 dB | 14 | 6 | 1000 Hz | 20.83 Hz |
| 5G NR 100MHz | 48.9 dB | 32 | 24 | 2000 Hz | 41.67 Hz |

출처 [^8]


## 이 사슬 위에서 무엇이 결정되나

2단계의 소거 깊이와 그 대가는 [편 52 «탭을 늘리면 환경이 정한 바닥에서 멈추고»](52_eca.ipynb) 가, 4단계 문턱의 눈금은 [편 53 «운용 형상에서 경험 Pfa 를 재니 명목값의…»](53_cfar-calib.ipynb) 가 든다. 3단계가 만드는 응답의 모양은 [편 49 «검출기가 실제로 쓰는 커널 그대로 모호함수를…»](49_ambiguity.ipynb) 가 이미 쟀다.

세 파형이 같은 코드 경로를 지나므로, 뒤 편들이 재는 격차는 사슬 차이가 아니라 파형 차이다.


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 회전 블레이드 산란을 이 사슬에 넣는 조건을 세운다 | 마이크로도플러를 검출 판정에 쓰는 조건이 결정된다 | [편 43 «상시 기준신호가 주는 것은 날개끝 확산이 아니…»](43_md-prf.ipynb) |
| 2채널 수신을 실제 X410 캡처로 바꿔 같은 사슬을 돌린다 | 시뮬 사슬과 실측 사슬이 같은 형상 표 위에 선다 | [편 67 «X410 의 12-bit ADC 동적범위가 직…»](67_hardware.ipynb) |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 8개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^1] | `outputs/verify_eca.json` | `meta.setups[1].dnr_db` | 42.99 |
| [^2] | `outputs/verify_eca.json` | `meta.setups[2].dnr_db` | 59.99 |
| [^3] | `outputs/verify_eca.json` | `meta.setups[0].dnr_db` | 48.85 |
| [^4] | `outputs/verify_eca.json` | `meta.setups[1].n_taps` | 24 |
| [^5] | `outputs/verify_eca.json` | `meta.setups[2].n_taps` | 14 |
| [^6] | `outputs/verify_eca.json` | `meta.setups[0].n_taps` | 32 |
| [^7] | `outputs/verify_cfar.json` | `meta.M_cpi` | 48 |
| [^8] | `outputs/verify_eca.json` | `meta.setups` | (3행 표) |


---

## 절 2. 탭을 늘리면 환경이 정한 바닥에서 멈추고, 그 대가가 0-도플러 노치다



> ### 한 일
> **ECA 탭 수를 1~96 으로 스윕하며 소거 깊이를 재고, 같은 소거기가 표적의 느린 도플러를 얼마나 함께 지우는지를 속도 문턱으로 환산했다.**

### 결과
1. 직접파만 든 신호에 같은 소거기를 걸면 float64 한계까지 내려간다 — 5G 232.3 dB [^9]. 측정된 다중경로를 넣으면 56.1 dB [^10] 에서 멈춘다.
2. 바닥을 정하는 것은 탭 수가 아니라 환경이다 — 탭 1~96 스윕에서 깊이가 포화한다.
3. 대가는 0-도플러 노치다. 3 dB 손실 지점은 $f_d/\Delta f_d$ = 0.596 [^11] 이고 세 파형이 같다.
4. 프레임 48 [^12]개에서 속도 문턱은 WiFi 0.39 m/s [^13] · LTE 1.10 m/s [^14] · 5G 1.16 m/s [^15] 다 — 그 위 속도는 온전히 남는다.
5. 정적 산란체는 ECA 뒤에서 죽은 파라미터다 — 클러터를 100 [^16]배까지 키워도 SCR 변화폭은 3.5e-09 dB [^17] 다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 소거기 | CPI 1회 최소제곱 사영의 standard ECA — 레퍼런스의 지연 사본이 치는 부분공간에 서베일런스를 통째로 사영한다(`src/passive_process.py:13`) |
| 깊이 스윕 | 탭 수 1~96 × (직접파만 / 측정된 다중경로 포함) 두 조건. 두 조건의 차이가 «바닥을 무엇이 정하는가» 를 가른다 |
| 노치 환산 | 3 dB 손실 지점을 $f_d/\Delta f_d$ 무차원으로 재고, 파형별 $\lambda$ 로 속도 문턱으로 옮긴다 |
| 클러터 대조 | 정적 산란체 세기를 배수로 키우며 SCR 변화폭을 잰다 |

### 재현

```bash
cd /home/yunjung/workspace/sionna2
PYTHONPATH=src ~/.venvs/py312/bin/python benchmark/verify_eca.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/viz_report04_detector.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/build_part09_detector.py
```

| | |
|---|---|
| 출력 | `outputs/verify_eca.json` |
| 소요 | ECA 검증 · 그림은 각각 수 분 (GPU 1장) |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| [편 51 «수신 → ECA → 거리도플러 → CFAR»](51_chain.ipynb) | 사슬의 네 단계와 파형별 형상 |

---


## 직접파를 얼마나 지울 수 있는가

탭을 늘리면 소거가 깊어지다가 멈춘다. 멈추는 자리를 정하는 것이 탭 수인지 환경인지를 가르려고 두 조건을 나란히 돌렸다.

| 파형 | 직접파만 | 직접파 + 다중경로(포화) |
|---|---|---|
| WiFi | 202.7 dB [^18] | 33.0 dB [^19] |
| LTE | 219.9 dB [^20] | 41.1 dB [^21] |
| 5G | 232.3 dB [^9] | 56.1 dB [^10] |

왼쪽은 float64 산술의 한계이고, 오른쪽이 실제 바닥이다. 두 열의 간격이 곧 **환경이 정하는 몫**이다.


![f2_eca_depth](../outputs/figures/report04_f2_eca_depth.png)

**그림 1.** ECA 소거 깊이의 바닥을 정하는 것은 무엇인가?


## 대가 — 0-도플러 노치

ECA 는 지연만 다른 성분을 함께 지운다. 느리게 움직이는 표적은 그 성분과 구분되지 않으므로 같이 깎인다. 3 dB 손실 지점은 $f_d/\Delta f_d$ = 0.596 [^11] 이고 세 파형이 같다 — 속도 문턱은 $\lambda$ 가 가른다.

| 파형 | 3 dB 속도 문턱 (프레임 48 [^12]개) |
|---|---|
| WiFi | 0.39 m/s [^13] |
| LTE | 1.10 m/s [^14] |
| 5G | 1.16 m/s [^15] |


![f3_eca_notch](../outputs/figures/report04_f3_eca_notch.png)

**그림 2.** ECA 가 클러터와 함께 지우는 표적의 속도는 얼마인가?


## 정적 클러터는 ECA 뒤에서 죽은 파라미터다

클러터 세기를 100 [^16]배까지 키워도 SCR 변화폭은 3.5e-09 dB [^17] 다. 소거기가 직접파와 함께 정적 성분을 통째로 가져가기 때문이다.

그래서 이 사슬에서 남는 위협은 정적 클러터가 아니라 **표적을 거쳐 오는 성분**이고, 느린 표적은 노치가 먼저 지운다 — 그 축의 결과는 [편 62 «CPI 를 늘리면 세 파형 모두 블라인드율이…»](62_cpi-sweep.ipynb) 가 든다.


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 실외에서 잰 채널로 소거 바닥을 다시 잰다 | 환경이 정하는 바닥이 실측 채널에서 몇 dB 인지 확정된다 | `benchmark/verify_eca.py` → [편 67 «X410 의 12-bit ADC 동적범위가 직…»](67_hardware.ipynb) |
| 노치 폭을 CPI 와 함께 스윕한다 | 느린 표적이 노치 밖으로 나오는 CPI 가 수치로 정해진다 | `benchmark/verify_eca.py` → [편 62 «CPI 를 늘리면 세 파형 모두 블라인드율이…»](62_cpi-sweep.ipynb) |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 13개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^9] | `outputs/verify_eca.json` | `S1_depth_vs_taps[0].rows[12].depth_dpi_db` | 232.3 |
| [^10] | `outputs/verify_eca.json` | `S1_depth_vs_taps[0].rows[12].depth_full_db` | 56.07 |
| [^11] | `outputs/verify_eca.json` | `S4_target_loss[1].fd_3db_over_dfd` | 0.5957 |
| [^12] | `outputs/verify_eca.json` | `S4_target_loss[4].M` | 48 |
| [^13] | `outputs/verify_eca.json` | `S4_target_loss[4].v_3db_ms` | 0.3906 |
| [^14] | `outputs/verify_eca.json` | `S4_target_loss[7].v_3db_ms` | 1.104 |
| [^15] | `outputs/verify_eca.json` | `S4_target_loss[1].v_3db_ms` | 1.163 |
| [^16] | `outputs/verify_eca.json` | `S5_clutter_dead.sweep[3].scale` | 100 |
| [^17] | `outputs/verify_eca.json` | `S5_clutter_dead.scr_span_db` | 3.539e-09 |
| [^18] | `outputs/verify_eca.json` | `S1_depth_vs_taps[1].rows[12].depth_dpi_db` | 202.7 |
| [^19] | `outputs/verify_eca.json` | `S1_depth_vs_taps[1].rows[12].depth_full_db` | 33.01 |
| [^20] | `outputs/verify_eca.json` | `S1_depth_vs_taps[2].rows[12].depth_dpi_db` | 219.9 |
| [^21] | `outputs/verify_eca.json` | `S1_depth_vs_taps[2].rows[12].depth_full_db` | 41.12 |


---

## 절 3. 운용 형상에서 경험 Pfa 를 재니 명목값의 1.52~2.66 배였다



> ### 한 일
> **운용 형상의 검출 사슬에서 거리-도플러 맵을 대량으로 다시 만들어 경험적 오경보율을 세고, 세 파형의 CFAR 문턱을 그 측정값에 맞춰 교정했다.**

### 결과
1. GPU 2717 s [^22] 동안 파형·모드마다 거리-도플러 맵 10,000 [^23]장을 돌려 경험 Pfa 를 측정했다.
2. 검출기 구현의 눈금을 먼저 확정했다 — 문턱 상수는 이론값과 상대오차 7.6e-16 [^24] 안에서 같고, 이상적 백색 맵 500,000 [^25]장(셀 564,000,000 [^26]개)에서 경험/명목 = 0.997 [^27] 다.
3. 운용 형상(CPI 프레임 48 [^28] · `g2x2_t6x6` · 0-도플러 마스크 1 [^29]빈)에서 명목 1e-04 [^30] 를 주면 WiFi 1.53 [^31]배 · LTE 2.66 [^32]배 · 5G 1.52 [^33]배로 울린다.
4. 그 형상의 교정표를 만들었다 — 경험 1e-04 [^30] 를 얻는 명목값은 WiFi 6.27e-05 [^34] · LTE 2.90e-05 [^35] · 5G 6.46e-05 [^36] 다.
5. 선행 census 16 [^37]편 · 전문 198 [^38]쪽에서 `CFAR` 와 `false alarm` 이 모두 0회인 논문이 13 [^39]편이고, 검출을 주장한 논문은 1 [^40]편이다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 경험 Pfa | 파형·명목값마다 거리-도플러 맵 10,000 [^23]장에 CA-CFAR 를 걸어 오경보 셀을 세었다 (GPU, 2717 s [^22]) |
| 문턱 상수 | CA-CFAR α 를 이론식과 대조했다 — 상대오차 7.6e-16 [^24] · 훈련셀 264 [^41]개 |
| 측정 구간 | 명목 1e-06 [^42] ~ 1e-02 [^43] 아홉 점. 그 밖의 운용점은 외삽이라 표에서 뺀다 |
| 교정표 | 측정한 명목–경험 곡선을 역보간한다. 자유공간 기하는 형상이 달라 `src/freespace_detect.py:711` 이 거기서 다시 잰다 |
| 왜 통제 시뮬레이션인가 | 오경보율을 명목값과 대조하려면 같은 배경을 수만 번 다시 만들어 세어야 한다. 실외 실측은 배경을 주어진 대로 받는다 |

### 재현

```bash
cd /home/yunjung/workspace/sionna2
PYTHONPATH=src ~/.venvs/py312/bin/python benchmark/verify_cfar.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/viz_report04_detector.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/build_part09_detector.py
```

| | |
|---|---|
| 출력 | `outputs/verify_cfar.json`, `outputs/prior_census.json` |
| 소요 | CFAR 측정이 2717 s [^22] (GPU 1장) |
| 비고 | 맵 수는 `--maps` / `--white` 로 줄인다. 줄이면 신뢰구간이 넓어진다. |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| [편 51 «수신 → ECA → 거리도플러 → CFAR»](51_chain.ipynb) | 사슬의 네 단계와 파형별 형상 |

---


## 왜 배경을 다시 만드는가

⭐ 오경보율을 명목값과 대조하려면 같은 배경을 수만 번 다시 만들어 세어야 한다. 통제 시뮬레이션이 그 일을 한다.

선행 census 16 [^37]편 · 전문 198 [^38]쪽에서 `CFAR` 와 `false alarm` 이 모두 0회인 논문이 13 [^39]편이고, 검출을 주장한 논문은 1 [^40]편이다. OpenISAC(`arXiv:2601.03535v2`, preprint)은 전문 16 [^44]쪽에서 `CFAR` 0 [^45]회 · `false alarm` 0 [^46]회 · `detection probability` 0 [^47]회다.


## 눈금부터 확정한다

이 절의 모든 수는 **운용 형상** 하나에서 나온다 — DPI+ECA · 운용 거리창 · CPI 프레임 48 [^28] · 훈련창 `g2x2_t6x6` · 0-도플러 마스크 1 [^29]빈.

검출기 구현이 먼저 맞아야 배율을 사슬 탓으로 돌릴 수 있다. 문턱 상수는 이론값과 상대오차 7.6e-16 [^24] 안에서 같고, 잡음 추정/실제 전력 = 1.000 [^48] 다. 이상적 백색 맵 500,000 [^25]장에서 경험/명목 = 0.997 [^27] 로 눈금이 1 에 선다.


![f4_pfa](../outputs/figures/report04_f4_pfa.png)

**그림 1.** 명목 Pfa 를 요구하면 실제로는 몇 배가 울리는가?


## 운용 형상 교정표

운용 명목값은 1e-04 [^30] 다. 왼쪽 열이 그 값에서 측정된 배율이고, 오른쪽 열이 교정된 명목값이다.

| 파형 | 명목 1e-4 에서 경험/명목 | 경험 1e-4 를 얻을 명목 Pfa |
|---|---|---|
| WiFi | 1.53 [^31]배 | 6.27e-05 [^34] |
| LTE | 2.66 [^32]배 | 2.90e-05 [^35] |
| 5G | 1.52 [^33]배 | 6.46e-05 [^36] |

세 파형의 배율이 서로 다르다. 교정이 셋을 같은 실제 오경보율 위에 올린다.

`src/passive_process.py:283` 이 이 JSON 을 읽고, `pfa_nominal_for()`(`src/passive_process.py:338`)가 파형별 명목값을 돌려준다.


## 이 표를 읽는 곳

`src/experiment_detection.py:358` 과 `src/experiment_x410.py:175` 가 `src/passive_process.py:283` 을 거쳐 이 표를 읽는다.

교정이 없으면 세 파형 비교가 서로 다른 실제 오경보율 위에서 이뤄진다. 그 위에서 선 비교가 [편 58 «자유공간 형상에서 문턱을 다시 재니 세 밴드가 SNR90 하나를 공유한다»](58_shared-threshold.ipynb) 다.

배율이 왜 1 이 아닌지, 이 표가 어디까지 쓰이는지는 [편 54 «그 배율의 원인은 셀 상관이고, 교정표는 형상마다 다시 재야 한다»](54_cfar-why.ipynb) 가 든다.


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 실외 클러터 배경 위에서 같은 Pfa 스윕을 돌린다 | 교정 배율이 배경에 따라 얼마나 움직이는지 수치로 확정된다 | `benchmark/verify_cfar.py` → [편 67 «X410 의 12-bit ADC 동적범위가 직…»](67_hardware.ipynb) |
| 맵 수를 한 자릿수 올려 명목 1e-06 [^49] 구간까지 측정한다 | 저 Pfa 운용점의 교정값이 측정 구간 안으로 들어온다 | `benchmark/verify_cfar.py --maps` · `calib_op_mask1.points` |
| 표적 σ 를 앵커 위에서 읽어 Pd 절대값을 다시 푼다 | Pd 절대값이 교정된 Pfa 와 같은 근거 위에 선다 | [편 60 «앵커 σ 위의 R90 은 비교가능 12칸에서…»](60_r90.ipynb) |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 28개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^22] | `outputs/verify_cfar.json` | `meta.runtime_s` | 2717 |
| [^23] | `outputs/verify_cfar.json` | `meta.n_maps_chain` | 10000 |
| [^24] | `outputs/verify_cfar.json` | `alpha_audit.g2x2_t6x6.rel_err` | 7.581e-16 |
| [^25] | `outputs/verify_cfar.json` | `meta.n_maps_white` | 500000 |
| [^26] | `outputs/verify_cfar.json` | `white.48x24.rows[89].cells` | 564000000 |
| [^27] | `outputs/verify_cfar.json` | `white.48x24.rows[89].ratio` | 0.9966 |
| [^28] | `outputs/verify_cfar.json` | `meta.M_cpi` | 48 |
| [^29] | `outputs/verify_cfar.json` | `meta.zd_mask_operational` | 1 |
| [^30] | `outputs/verify_cfar.json` | `chain.NR100.dpi_eca.op.rows[89].pfa_nom` | 0.0001 |
| [^31] | `outputs/verify_cfar.json` | `chain.WiFi80.dpi_eca.op.rows[89].ratio` | 1.531 |
| [^32] | `outputs/verify_cfar.json` | `chain.LTE20.dpi_eca.op.rows[89].ratio` | 2.663 |
| [^33] | `outputs/verify_cfar.json` | `chain.NR100.dpi_eca.op.rows[89].ratio` | 1.521 |
| [^34] | `outputs/verify_cfar.json` | `chain.WiFi80.dpi_eca.calib_op_mask1.points[2].pfa_nominal_needed` | 6.27e-05 |
| [^35] | `outputs/verify_cfar.json` | `chain.LTE20.dpi_eca.calib_op_mask1.points[2].pfa_nominal_needed` | 2.905e-05 |
| [^36] | `outputs/verify_cfar.json` | `chain.NR100.dpi_eca.calib_op_mask1.points[2].pfa_nominal_needed` | 6.46e-05 |
| [^37] | `outputs/prior_census.json` | `meta.n_papers` | 16 |
| [^38] | `outputs/prior_census.json` | `counts.total_pages` | 198 |
| [^39] | `outputs/prior_census.json` | `counts.zero_cfar_and_falsealarm` | 13 |
| [^40] | `outputs/prior_census.json` | `counts.claims_detection` | 1 |
| [^41] | `outputs/verify_cfar.json` | `alpha_audit.g2x2_t6x6.N_interior` | 264 |
| [^42] | `outputs/verify_cfar.json` | `meta.pfa_nominal[8]` | 1e-06 |
| [^43] | `outputs/verify_cfar.json` | `meta.pfa_nominal[0]` | 0.01 |
| [^44] | `outputs/prior_census.json` | `papers[14].pages` | 16 |
| [^45] | `outputs/prior_census.json` | `papers[14].terms.cfar` | 0 |
| [^46] | `outputs/prior_census.json` | `papers[14].terms.false_alarm` | 0 |
| [^47] | `outputs/prior_census.json` | `papers[14].terms.detection_probability` | 0 |
| [^48] | `outputs/verify_cfar.json` | `alpha_audit.g2x2_t6x6.noise_est_over_power` | 1 |
| [^49] | `outputs/verify_cfar.json` | `chain.NR100.dpi_eca.calib_op_mask1.points[4].pfa_target_emp` | 1e-06 |


---

## 절 4. 그 배율의 원인은 셀 상관이고, 교정표는 형상마다 다시 재야 한다



> ### 한 일
> **명목과 경험 사이의 배율을 만드는 항을 대조군 두 종으로 하나씩 꺼 확정하고, 그 배율이 형상에 얼마나 의존하는지를 창 폭을 바꿔 재었다.**

### 결과
1. CA-CFAR 는 훈련셀이 서로 독립이라고 가정한다. 사슬은 slow-time Hann 창으로 도플러축 셀을 묶는다 — Hann 을 rect 창으로 바꾸면 5G 배율이 0.96 [^50] 로 내려온다.
2. 거리축 항까지 끄면 1.02 [^51] 로 눈금이 1 로 돌아온다 — 두 항이 배율의 전부다.
3. 형상이 배율을 정한다 — 같은 파형이 운용 창에서 1.52 [^52]배, 넓은 창(256 [^53] 빈)에서 47.70 [^54]배다.
4. 그래서 교정표는 형상마다 다시 잰다. `check_detector_config()`(`src/passive_process.py:383`)가 거리창과 0-도플러 마스크 두 조건을 강제한다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 대조군 2종 | slow-time Hann 제거(도플러축) · 백색화 정합필터(거리축). 하나씩 끄고 배율이 어디로 가는지를 본다 |
| 사다리 읽기 | 이상적 백색 맵 → 잡음 맵 → 항을 하나씩 뺀 대조군 → 전체 사슬 순서로 읽으면 각 항의 몫이 갈린다 |
| 형상 의존 | 운용 창과 넓은 창(256 [^53] 빈)에서 같은 파형을 다시 재 배율의 형상 의존을 크기로 적는다 |
| 두 형상의 쓰임 | 운용 형상의 표는 챔버 기하가 읽고, 자유공간 기하는 `src/freespace_detect.py:711` 이 자기 형상에서 다시 잰다 |

### 재현

```bash
cd /home/yunjung/workspace/sionna2
PYTHONPATH=src ~/.venvs/py312/bin/python benchmark/verify_cfar.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/viz_report04_detector.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/build_part09_detector.py
```

| | |
|---|---|
| 출력 | `outputs/verify_cfar.json` |
| 소요 | CFAR 측정이 2717 s [^55] (GPU 1장) |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| [편 53 «운용 형상에서 경험 Pfa 를 재니 명목값의…»](53_cfar-calib.ipynb) | 운용 형상에서 잰 경험 Pfa 와 교정표 |

---


## 원인 — 셀 상관

CA-CFAR 는 훈련셀이 서로 독립이라고 가정한다. 사슬은 slow-time Hann 창으로 도플러축 셀을 묶고, 정합필터로 거리축 셀을 묶는다. 대조군이 그 두 항을 원인으로 확정한다(5G NR, 명목 1e-4).

| 조건 | 경험/명목 |
|---|---|
| 이상적 백색 맵 | 0.997 [^56] |
| 잡음 맵 (Hann + 정합필터) | 1.25 [^57] |
|   └ Hann 제거 (rect 창) | 0.96 [^50] |
|   └ 백색화 정합필터 (거리축 평탄) | 1.25 [^58] |
|   └ 둘 다 제거 | 1.02 [^51] |
| 전체 사슬 (직접파 + ECA) | 1.52 [^52] |


![f5_cause](../outputs/figures/report04_f5_cause.png)

**그림 1.** 명목과 경험 사이의 배율을 만드는 것은 무엇인가?


## 셀 상관은 어느 검출기에나 있다 — 그래서 대조군이 필요하다

«상관이 있다» 만으로는 원인 지목이 서지 않는다. 항을 하나씩 꺼서 눈금이 1 로 돌아오는 것을 보여야 확정된다. Hann 을 rect 로 바꾸면 0.96 [^50], 백색화 정합필터까지 끄면 1.02 [^51] 다.

그 사다리가 위 표이고, 마지막 줄과 첫 줄 사이의 간격이 곧 두 항의 몫이다.


## 형상 규약 — 교정표가 성립하는 조건

거리창은 ECA 탭 안에 두고, 0-도플러 행 1 [^59]개를 마스킹한다. `check_detector_config()`(`src/passive_process.py:383`)가 두 조건을 검사한다.

| 파형 | 운용 창 | 넓은 창 |
|---|---|---|
| WiFi | 1.53 [^60]배 | 41.1 [^61]배 |
| LTE | 2.66 [^62]배 | 58.8 [^63]배 |
| 5G | 1.52 [^52]배 | 47.7 [^54]배 |

창을 256 [^53] 빈으로 넓히면 배율이 두 자릿수가 된다. 교정표는 운용 창 형상에서 측정한 값이다.


## 어느 형상의 교정표가 어디에 쓰이나

| 형상 | 무엇을 재나 | 재는 코드 | 그 값을 읽는 곳 |
|---|---|---|---|
| 운용 형상 — CPI 프레임 48 [^64] · `g2x2_t6x6` · 마스크 1 [^59]빈 · 운용 거리창 | 이 부의 교정표 | `benchmark/verify_cfar.py` | `src/experiment_detection.py:358` · `src/experiment_x410.py:175` |
| 자유공간 형상 — 모드별 프레임 수 · 자유공간 거리창 · 0-도플러 가드 | 자유공간 명목 Pfa | `src/freespace_detect.py:711` | `src/experiment_freespace_range.py:206` |

[편 58 «자유공간 형상에서 문턱을 다시 재니 세 밴드가…»](58_shared-threshold.ipynb) 가 싣는 명목 Pfa 는 둘째 줄에서 나온 수다 — 이 부의 교정표와 형상이 달라 값도 다르다.


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 훈련셀에서도 0-도플러 행을 빼는 CFAR 변형을 만든다 | 넓은 창에서 마스크 폭 3 이 만드는 배율 0.65 [^65]배가 마스크 폭과 분리된다 | `src/passive_process.py:352` |
| 표적모형 민감도를 이 배율 위에서 다시 푼다 | 세 표적모형의 절대 소요이득이 경험 Pfa 위에 선다 — CA-CFAR 문턱은 세 팔에 같은 오프셋을 주므로 모형 간 차이는 문턱 규약에 불변이다 | [편 65 «평판·큐브·우리 격자를 같은 동작점에서 갈아끼…»](65_target-model-swap.ipynb) |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 16개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^50] | `outputs/verify_cfar.json` | `control_rect_window_NR100.op.rows[89].ratio` | 0.9619 |
| [^51] | `outputs/verify_cfar.json` | `control_whitened_mf_rect_NR100.op.rows[89].ratio` | 1.02 |
| [^52] | `outputs/verify_cfar.json` | `chain.NR100.dpi_eca.op.rows[89].ratio` | 1.521 |
| [^53] | `outputs/verify_cfar.json` | `meta.n_range_wide` | 256 |
| [^54] | `outputs/verify_cfar.json` | `chain.NR100.dpi_eca.wide.rows[89].ratio` | 47.7 |
| [^55] | `outputs/verify_cfar.json` | `meta.runtime_s` | 2717 |
| [^56] | `outputs/verify_cfar.json` | `white.48x24.rows[89].ratio` | 0.9966 |
| [^57] | `outputs/verify_cfar.json` | `chain.NR100.noise.op.rows[89].ratio` | 1.246 |
| [^58] | `outputs/verify_cfar.json` | `control_whitened_mf_NR100.op.rows[89].ratio` | 1.246 |
| [^59] | `outputs/verify_cfar.json` | `meta.zd_mask_operational` | 1 |
| [^60] | `outputs/verify_cfar.json` | `chain.WiFi80.dpi_eca.op.rows[89].ratio` | 1.531 |
| [^61] | `outputs/verify_cfar.json` | `chain.WiFi80.dpi_eca.wide.rows[89].ratio` | 41.14 |
| [^62] | `outputs/verify_cfar.json` | `chain.LTE20.dpi_eca.op.rows[89].ratio` | 2.663 |
| [^63] | `outputs/verify_cfar.json` | `chain.LTE20.dpi_eca.wide.rows[89].ratio` | 58.8 |
| [^64] | `outputs/verify_cfar.json` | `meta.M_cpi` | 48 |
| [^65] | `outputs/verify_cfar.json` | `chain.LTE20.dpi_eca.wide.rows[90].ratio` | 0.6538 |
